# Semana 12: Modelos Avanzados de Clasificación

## Caso Práctico: Sistema de Selección de Titulares del FC Barcelona

Imagina que trabajas como analista de datos para el FC Barcelona. El cuerpo técnico necesita un sistema que les ayude a decidir qué jugadores deberían ser titulares basándose en sus estadísticas de la temporada. Para hacer esta decisión más robusta, quieren combinar las opiniones de diferentes modelos predictivos, similar a cómo un director técnico consulta con varios especialistas antes de decidir la alineación.

**Objetivo del proyecto:** Desarrollar y combinar múltiples modelos de clasificación para predecir si un jugador debería ser titular o suplente.

**Contexto:** El Barcelona tiene una plantilla de 15 jugadores y necesita determinar quiénes deberían ser titulares considerando tres aspectos: rendimiento goleador, experiencia (minutos jugados), y edad.

---

## SESIÓN 1: Preparación de Datos y Primer Modelo (50 min)

Esta sesión cubrirá dos módulos usando el enfoque T.A.C.O.

### Módulo 1: Carga del Dataset de Jugadores

**Tarea**: Cargar un conjunto de datos de jugadores del Barcelona desde un archivo CSV que contiene estadísticas de la temporada y el estatus de cada jugador como titular o suplente.

**Enfoque**: En proyectos reales de ciencia de datos, la generación/recolección de datos y el análisis están separados. Esta separación tiene varias ventajas profesionales:

**¿Por qué separar generación de datos del análisis?**
1. **Reutilización**: El mismo dataset puede usarse en múltiples notebooks sin duplicar código
2. **Colaboración**: El equipo de datos genera archivos, el equipo de análisis los consume
3. **Reproducibilidad**: Todos trabajan con exactamente los mismos datos base
4. **Claridad**: Cada script tiene una responsabilidad única (Single Responsibility Principle)
5. **Versionado**: Los archivos CSV pueden versionarse y compartirse fácilmente

**Dataset disponible: datos_barcelona.csv**

Este archivo fue generado previamente ejecutando el script `generar_datos_liga.py`, que crea datos sintéticos con distribuciones estadísticas realistas del fútbol profesional:
- **Goles**: Distribución exponencial (0-15 goles, mayoría marca poco)
- **Minutos**: Distribución uniforme (400-2200 minutos)
- **Edad**: Distribución normal centrada en 25 años (18-35 años)
- **Es_Titular**: Calculado mediante score: 60% minutos + 40% goles

Usaremos pandas para leer el archivo CSV y cargarlo en un DataFrame para análisis.

**Código**:

In [1]:
# Importamos las librerías necesarias para el análisis
import pandas as pd  # Manipulación de datos
import numpy as np  # Operaciones numéricas
from sklearn.model_selection import train_test_split  # División de datos
from sklearn.linear_model import LogisticRegression  # Modelo de regresión logística
from sklearn.ensemble import RandomForestClassifier  # Modelo de bosque aleatorio
from sklearn.metrics import accuracy_score  # Métrica de evaluación
import warnings

# Suprimimos advertencias para mantener la salida limpia
warnings.filterwarnings("ignore")

# Cargamos el dataset desde el archivo CSV
# Este archivo fue generado previamente con el script generar_datos_liga.py
nombre_archivo = "datos_barcelona.csv"

try:
    # Intentamos cargar el archivo CSV
    datos_barca = pd.read_csv(nombre_archivo, encoding='utf-8')
    print(f"Dataset cargado exitosamente desde: {nombre_archivo}")
    print("="*70)
    
except FileNotFoundError:
    # Si el archivo no existe, mostramos mensaje de error útil
    print(f"ERROR: No se encontró el archivo '{nombre_archivo}'")
    print("\nPara generar el archivo, ejecuta primero:")
    print("  python generar_datos_liga.py")
    print("\nEsto creará el archivo CSV con datos sintéticos del Barcelona.")
    raise

# Mostramos el dataset completo
print("\nDataset de jugadores del Barcelona:")
print(datos_barca.to_string(index=False))

# Mostramos estadísticas resumen
print("\n" + "="*70)
print("Resumen estadístico del dataset:")
print(f"  Total de jugadores: {len(datos_barca)}")
print(f"  Titulares: {datos_barca['Es_Titular'].sum()}")
print(f"  Suplentes: {len(datos_barca) - datos_barca['Es_Titular'].sum()}")
print(f"  Goles promedio: {datos_barca['Goles_Temporada'].mean():.1f}")
print(f"  Minutos promedio: {datos_barca['Minutos_Jugados'].mean():.0f}")
print(f"  Edad promedio: {datos_barca['Edad'].mean():.1f} años")

Dataset cargado exitosamente desde: datos_barcelona.csv

Dataset de jugadores del Barcelona:
         Jugador  Goles_Temporada  Minutos_Jugados  Edad  Es_Titular
       Defensa_1                5             2169    23           1
 Mediocampista_7                0             2095    22           1
 Mediocampista_3                9             1947    27           1
      Defensa_15                0             1899    28           1
      Portero_13                0             1629    20           1
       Portero_4                1             1624    21           1
    Delantero_14                1             1497    30           1
 Mediocampista_8                1             1413    25           1
       Portero_2                0             1240    19           0
       Portero_5                1             1210    18           0
 Mediocampista_9                1             1093    24           0
Mediocampista_12                0              833    24           0
      Defe

**Resultado**:

El código ha cargado exitosamente el dataset desde el archivo CSV `datos_barcelona.csv`. Este enfoque de separar la generación de datos del análisis es una **práctica profesional estándar** en ciencia de datos.

**Ventajas observadas al cargar desde CSV:**

1. **Workflow realista**: En proyectos reales, raramente generamos datos en el mismo notebook donde hacemos análisis. Los datos vienen de bases de datos, APIs, o archivos compartidos.

2. **Reproducibilidad garantizada**: Todos los estudiantes y el profesor trabajan con exactamente el mismo dataset, eliminando variaciones por diferentes generaciones aleatorias.

3. **Separación de responsabilidades**:
   - **generar_datos_liga.py**: Script dedicado solo a generar datos sintéticos
   - **modelos-avanzados-clasificacion.ipynb**: Notebook enfocado solo en análisis y modelado

4. **Reutilización eficiente**: Si necesitáramos este dataset en otro notebook (ej: visualización avanzada, análisis exploratorio), simplemente cargamos el mismo CSV sin duplicar código de generación.

5. **Control de versiones**: El archivo CSV puede ser versionado con Git, permitiendo rastrear cambios en los datos a lo largo del tiempo.

**Observaciones del dataset cargado:**
- 15 jugadores distribuidos entre titulares y suplentes
- Los jugadores están ordenados por minutos jugados (de mayor a menor)
- Estadísticas realistas que reflejan patrones del fútbol profesional
- Distribución de ~60% titulares y ~40% suplentes (proporción realista)

**Manejo de errores implementado:**
- El código incluye try/except para manejar el caso donde el CSV no existe
- Proporciona instrucciones claras sobre cómo generar el archivo faltante
- Esto enseña prácticas de programación defensiva

Este dataset será la base para entrenar nuestros modelos predictivos. El siguiente paso es preparar los datos dividiéndolos en características (X) y objetivo (y).

### Módulo 2: Preparación de Datos para Entrenamiento

**Tarea**: Dividir el dataset en características predictoras (X) y variable objetivo (y), luego separar los datos en conjuntos de entrenamiento y prueba.

**Enfoque**: Para entrenar modelos de machine learning necesitamos separar nuestros datos en dos componentes:

1. **Variables predictoras (X)**: Las características que usaremos para hacer predicciones (Goles_Temporada, Minutos_Jugados, Edad)
2. **Variable objetivo (y)**: Lo que queremos predecir (Es_Titular)

Además, dividiremos estos datos en dos conjuntos:
- **Conjunto de entrenamiento (70%)**: Datos que el modelo usará para aprender patrones
- **Conjunto de prueba (30%)**: Datos nuevos que el modelo nunca ha visto, para evaluar su capacidad de generalización

Esta división es crucial porque queremos saber si nuestro modelo puede predecir correctamente el estatus de jugadores que no estaban en los datos de entrenamiento, simulando situaciones reales donde lleguen nuevos jugadores al equipo.

**Código**:

In [2]:
# Separamos el dataset en características (X) y objetivo (y)
# X contiene las tres variables que usaremos para predecir
X = datos_barca[["Goles_Temporada", "Minutos_Jugados", "Edad"]]

# y contiene la variable que queremos predecir (titular o suplente)
y = datos_barca["Es_Titular"]

print("Características predictoras (X):")
print(X.head())
print(f"\nDimensiones de X: {X.shape[0]} jugadores, {X.shape[1]} características")

print("\n" + "="*60)
print("Variable objetivo (y):")
print(y.head())
print(f"\nDistribución: {sum(y)} titulares, {len(y) - sum(y)} suplentes")

# Dividimos los datos en entrenamiento (70%) y prueba (30%)
# random_state=42 asegura resultados reproducibles
X_entrenamiento, X_prueba, y_entrenamiento, y_prueba = train_test_split(
    X, y, test_size=0.3, random_state=42
)

print("\n" + "="*60)
print("División de datos completada:")
print(f"Conjunto de entrenamiento: {len(X_entrenamiento)} jugadores ({len(X_entrenamiento)/len(X)*100:.1f}%)")
print(f"Conjunto de prueba: {len(X_prueba)} jugadores ({len(X_prueba)/len(X)*100:.1f}%)")

Características predictoras (X):
   Goles_Temporada  Minutos_Jugados  Edad
0                5             2169    23
1                0             2095    22
2                9             1947    27
3                0             1899    28
4                0             1629    20

Dimensiones de X: 15 jugadores, 3 características

Variable objetivo (y):
0    1
1    1
2    1
3    1
4    1
Name: Es_Titular, dtype: int64

Distribución: 9 titulares, 6 suplentes

División de datos completada:
Conjunto de entrenamiento: 10 jugadores (66.7%)
Conjunto de prueba: 5 jugadores (33.3%)


**Resultado**:

Los datos han sido preparados exitosamente. Tenemos:

- **Características (X)**: 15 jugadores con 3 variables cada uno (Goles, Minutos, Edad)
- **Objetivo (y)**: 9 titulares y 6 suplentes en total

La división train/test ha creado:
- **Entrenamiento**: 10 jugadores (70%) - estos datos se usarán para enseñar a los modelos
- **Prueba**: 5 jugadores (30%) - estos datos se usarán para evaluar qué tan bien predicen los modelos

Esta separación es análoga a cómo un entrenador aprende de partidos pasados (entrenamiento) y luego aplica ese conocimiento en nuevos partidos (prueba). Los modelos aprenderán patrones de los 10 jugadores de entrenamiento y luego intentarán predecir correctamente el estatus de los 5 jugadores que nunca han "visto" antes.

Con los datos preparados, estamos listos para entrenar nuestro primer modelo en la siguiente sesión.

---

## SESIÓN 2: Entrenamiento de Múltiples Modelos (50 min)

Esta sesión desarrollará dos módulos: entrenar tres modelos diferentes y comparar sus predicciones.

### Módulo 3: Entrenamiento de Tres Modelos de Clasificación

**Tarea**: Entrenar tres modelos diferentes (Regresión Logística, Random Forest y un segundo modelo de Regresión Logística como comparación) usando los datos de entrenamiento preparados en la sesión anterior.

**Enfoque**: Así como un director técnico consulta con diferentes especialistas (preparador físico, médico deportivo, analista táctico), vamos a crear tres modelos que "analizan" los datos desde diferentes perspectivas matemáticas:

1. **Modelo 1 - Regresión Logística**: Busca una relación lineal entre las variables. Es como un especialista que asume que "más goles + más minutos + edad óptima = titular". Es rápido y fácil de interpretar.

2. **Modelo 2 - Random Forest (Bosque Aleatorio)**: Crea múltiples "árboles de decisión" que votan. Es como tener un comité de mini-especialistas que consideran diferentes combinaciones de factores. Es más complejo pero puede captar relaciones no lineales.

3. **Modelo 3 - Segunda Regresión Logística**: Entrenamos otro modelo del mismo tipo para verificar consistencia. En situaciones reales, a veces entrenamos el mismo tipo de modelo con diferentes configuraciones.

El proceso de entrenamiento (.fit()) es cuando el modelo "estudia" los datos históricos para aprender los patrones que distinguen titulares de suplentes.

**Código**:

In [3]:
# Creamos e instanciamos los tres modelos

# Modelo 1: Regresión Logística (perspectiva lineal)
modelo_logistico_1 = LogisticRegression(random_state=42)
# Entrenamos el modelo con los datos de entrenamiento
modelo_logistico_1.fit(X_entrenamiento, y_entrenamiento)
print("[OK] Modelo 1 (Regresión Logística) - Entrenamiento completado")

# Modelo 2: Random Forest (perspectiva no-lineal con votación interna)
# n_estimators=10 significa que creará 10 árboles de decisión
modelo_random_forest = RandomForestClassifier(n_estimators=10, random_state=42)
modelo_random_forest.fit(X_entrenamiento, y_entrenamiento)
print("[OK] Modelo 2 (Random Forest) - Entrenamiento completado")

# Modelo 3: Segunda Regresión Logística (para comparación)
modelo_logistico_2 = LogisticRegression(random_state=42)
modelo_logistico_2.fit(X_entrenamiento, y_entrenamiento)
print("[OK] Modelo 3 (Regresión Logística duplicado) - Entrenamiento completado")

print("\n" + "="*60)
print("Los tres modelos han completado su entrenamiento.")
print("Cada modelo ha 'aprendido' de los mismos 10 jugadores de entrenamiento,")
print("pero usando diferentes estrategias matemáticas.")

[OK] Modelo 1 (Regresión Logística) - Entrenamiento completado
[OK] Modelo 2 (Random Forest) - Entrenamiento completado
[OK] Modelo 3 (Regresión Logística duplicado) - Entrenamiento completado

Los tres modelos han completado su entrenamiento.
Cada modelo ha 'aprendido' de los mismos 10 jugadores de entrenamiento,
pero usando diferentes estrategias matemáticas.


**Resultado**:

Los tres modelos han sido entrenados exitosamente. Cada uno ha procesado los mismos 10 jugadores de entrenamiento, pero ha "aprendido" de manera diferente:

- **Modelo 1 (Regresión Logística)**: Ha encontrado coeficientes que relacionan linealmente goles, minutos y edad con la probabilidad de ser titular
- **Modelo 2 (Random Forest)**: Ha creado 10 árboles de decisión, cada uno considerando diferentes combinaciones de las variables
- **Modelo 3 (Regresión Logística duplicado)**: Debería dar resultados idénticos al Modelo 1 si usamos el mismo random_state

Es importante notar que hasta este momento los modelos solo han "estudiado" - aún no han hecho predicciones. El entrenamiento es como cuando un scout ve muchos partidos para aprender qué hace a un jugador titular, pero aún no ha evaluado a jugadores nuevos.

En el siguiente módulo haremos que cada modelo prediga el estatus de los 5 jugadores que guardamos para prueba, y veremos si sus "opiniones" difieren.

### Módulo 4: Comparación de Predicciones

**Tarea**: Obtener predicciones de los tres modelos para los 5 jugadores de prueba y compararlas para identificar similitudes y diferencias.

**Enfoque**: Una vez entrenados, los modelos pueden hacer predicciones (.predict()) sobre datos nuevos. Les daremos las características (goles, minutos, edad) de los 5 jugadores de prueba y cada modelo nos dirá si cree que deberían ser titulares (1) o suplentes (0).

La comparación de predicciones es fundamental porque:
1. Nos muestra si los modelos están de acuerdo (lo cual indica alta confianza)
2. Identifica casos difíciles donde los modelos discrepan
3. Nos permite ver si diferentes enfoques matemáticos llegan a las mismas conclusiones

Es como cuando tres scouts observan al mismo jugador - si los tres coinciden, la decisión es clara. Si difieren, necesitamos analizar más profundamente.

**Código**:

In [4]:
# Cada modelo genera predicciones para los 5 jugadores de prueba
predicciones_modelo1 = modelo_logistico_1.predict(X_prueba)
predicciones_modelo2 = modelo_random_forest.predict(X_prueba)
predicciones_modelo3 = modelo_logistico_2.predict(X_prueba)

# Creamos un DataFrame para visualizar y comparar las predicciones
tabla_comparacion = pd.DataFrame({
    "Jugador_ID": range(1, len(X_prueba) + 1),
    "Modelo_1_LogReg": predicciones_modelo1,
    "Modelo_2_RandomForest": predicciones_modelo2,
    "Modelo_3_LogReg": predicciones_modelo3,
    "Realidad": y_prueba.values
})

print("Comparación de predicciones por modelo:")
print("(1 = Titular, 0 = Suplente)")
print("\n" + tabla_comparacion.to_string(index=False))

# Calculamos el acuerdo entre los tres modelos
# Acuerdo total: los tres modelos predicen lo mismo
acuerdo_total = (
    (predicciones_modelo1 == predicciones_modelo2) & 
    (predicciones_modelo2 == predicciones_modelo3)
)
num_acuerdos = sum(acuerdo_total)

print("\n" + "="*60)
print(f"Los tres modelos coinciden en {num_acuerdos} de {len(acuerdo_total)} jugadores")
print(f"Porcentaje de acuerdo: {num_acuerdos/len(acuerdo_total)*100:.1f}%")

# Identificamos casos de desacuerdo
if num_acuerdos < len(acuerdo_total):
    print(f"\nCasos con desacuerdo:")
    desacuerdos = tabla_comparacion[~acuerdo_total]
    print(desacuerdos.to_string(index=False))

Comparación de predicciones por modelo:
(1 = Titular, 0 = Suplente)

 Jugador_ID  Modelo_1_LogReg  Modelo_2_RandomForest  Modelo_3_LogReg  Realidad
          1                0                      0                0         0
          2                0                      0                0         0
          3                1                      1                1         1
          4                0                      0                0         0
          5                1                      1                1         1

Los tres modelos coinciden en 5 de 5 jugadores
Porcentaje de acuerdo: 100.0%


**Resultado**:

La tabla de comparación muestra las predicciones de cada modelo para los 5 jugadores de prueba. Observaciones clave:

1. **Modelos 1 y 3 (ambos Regresión Logística)**: Deberían tener predicciones idénticas ya que son el mismo tipo de modelo entrenado con el mismo random_state

2. **Modelo 2 (Random Forest)**: Puede diferir de los modelos de Regresión Logística porque usa un enfoque completamente diferente (votación de árboles de decisión vs. regresión lineal)

3. **Concordancia**: Un alto porcentaje de acuerdo (>80%) indica que los patrones en los datos son claros y consistentes. Un bajo porcentaje sugiere que las decisiones son más ambiguas y dependen del enfoque matemático usado.

4. **Comparación con la realidad**: La columna "Realidad" nos muestra si el jugador es realmente titular o suplente, permitiéndonos evaluar qué predicciones son correctas.

Si hay desacuerdos, estos casos son particularmente interesantes - representan jugadores "en el límite" donde las características no indican claramente si deberían ser titulares. En la siguiente sesión combinaremos estas predicciones mediante votación para tomar decisiones más robustas.

---

## SESIÓN 3: Combinación de Predicciones y Evaluación Final (50 min)

Esta sesión final desarrollará dos módulos: combinar predicciones mediante votación y evaluar el rendimiento de todos los modelos.

### Módulo 5: Implementación de Votación por Mayoría

**Tarea**: Crear un sistema de votación que combine las predicciones de los tres modelos para tomar una decisión final más robusta.

**Enfoque**: La votación por mayoría (ensemble voting) es una técnica donde múltiples modelos "votan" y se elige la predicción más frecuente. Es similar a cómo funciona un cuerpo técnico en el fútbol:

- **Director Técnico**: "¿Debería jugar Ansu Fati?"
- **Scout Ofensivo (Modelo 1)**: "Sí" (1)
- **Analista Táctico (Modelo 2)**: "No" (0)
- **Scout Defensivo (Modelo 3)**: "Sí" (1)
- **Decisión Final**: "Sí" (2 votos a favor vs 1 en contra)

Implementaremos una función votacion_mayoria() que:
1. Suma las predicciones de los tres modelos para cada jugador
2. Si la suma es >= 2, la decisión final es "titular" (1)
3. Si la suma es < 2, la decisión final es "suplente" (0)

Esta estrategia reduce el riesgo de errores individuales ya que un modelo equivocado puede ser compensado por los otros dos.

**Código**:

In [5]:
def votacion_mayoria(pred1, pred2, pred3):
    """
    Combina tres predicciones binarias usando votación por mayoría simple.
    
    Parámetros:
    -----------
    pred1, pred2, pred3 : arrays de numpy
        Predicciones de los tres modelos (valores 0 o 1)
    
    Retorna:
    --------
    numpy array
        Predicción combinada (1 si mayoría dice titular, 0 si mayoría dice suplente)
    
    Lógica:
    -------
    - Suma los votos: 0+0+0=0, 0+0+1=1, 0+1+1=2, 1+1+1=3
    - Si suma >= 2: al menos 2 modelos dijeron "titular" (1)
    - Si suma < 2: la mayoría dijo "suplente" (0)
    """
    votos_totales = pred1 + pred2 + pred3
    decision_final = (votos_totales >= 2).astype(int)
    return decision_final


# Aplicamos la votación a las predicciones de nuestros tres modelos
prediccion_votacion = votacion_mayoria(
    predicciones_modelo1,
    predicciones_modelo2, 
    predicciones_modelo3
)

# Agregamos la decisión por votación a nuestra tabla de comparación
tabla_comparacion["Votacion_Mayoría"] = prediccion_votacion

print("Tabla completa con decisión por votación:")
print("\n" + tabla_comparacion.to_string(index=False))

print("\n" + "="*60)
print("Análisis de la votación:")
# Contamos cuántos jugadores fueron clasificados como titulares por votación
titulares_votacion = sum(prediccion_votacion)
print(f"Jugadores clasificados como titulares: {titulares_votacion} de {len(prediccion_votacion)}")
print(f"Jugadores clasificados como suplentes: {len(prediccion_votacion) - titulares_votacion} de {len(prediccion_votacion)}")

Tabla completa con decisión por votación:

 Jugador_ID  Modelo_1_LogReg  Modelo_2_RandomForest  Modelo_3_LogReg  Realidad  Votacion_Mayoría
          1                0                      0                0         0                 0
          2                0                      0                0         0                 0
          3                1                      1                1         1                 1
          4                0                      0                0         0                 0
          5                1                      1                1         1                 1

Análisis de la votación:
Jugadores clasificados como titulares: 2 de 5
Jugadores clasificados como suplentes: 3 de 5


**Resultado**:

La tabla ahora incluye una quinta columna "Votacion_Mayoría" que muestra la decisión combinada de los tres modelos. Observaciones importantes:

1. **Consenso vs. Desacuerdo**: 
   - Cuando los tres modelos coinciden, la votación simplemente confirma esa predicción unánime
   - Cuando hay desacuerdo (ej: 2 modelos dicen "titular", 1 dice "suplente"), la votación sigue a la mayoría

2. **Robustez**: La votación es más robusta que cualquier modelo individual porque:
   - Un solo modelo equivocado no compromete la decisión final
   - Se aprovechan las fortalezas de diferentes enfoques matemáticos
   - Reduce el impacto de casos límite o ambiguos

3. **Ejemplo práctico**: Si los modelos votaron [1, 0, 1] para un jugador, la suma es 2, que es >= 2, por lo tanto la decisión final es 1 (titular)

4. **Limitación importante**: La votación solo funciona bien si los modelos son suficientemente diferentes y competentes. Si todos los modelos tienen el mismo sesgo o error, la votación no ayudará.

El siguiente paso es evaluar numéricamente qué tan bien funciona cada modelo (incluyendo la votación) comparando sus predicciones con la realidad.

### Módulo 6: Evaluación Comparativa de Rendimiento

**Tarea**: Calcular y comparar la precisión (accuracy) de los tres modelos individuales y de la votación por mayoría para determinar cuál estrategia funciona mejor.

**Enfoque**: La precisión es la métrica más simple para evaluar clasificadores binarios:

**Precisión = (Predicciones Correctas) / (Total de Predicciones)**

Por ejemplo, si un modelo acierta en 4 de 5 jugadores, su precisión es 4/5 = 0.80 = 80%.

Compararemos cuatro estrategias:
1. **Modelo 1 (Regresión Logística)**: Enfoque lineal simple
2. **Modelo 2 (Random Forest)**: Enfoque con votación interna de árboles
3. **Modelo 3 (Regresión Logística duplicado)**: Debería ser idéntico al Modelo 1
4. **Votación por Mayoría**: Combinación de los tres modelos

La estrategia con mayor precisión será la que más se acerca a la "verdad" (los valores reales de Es_Titular). Sin embargo, con solo 5 jugadores de prueba, pequeñas diferencias pueden ser casualidad. En aplicaciones reales usaríamos conjuntos de prueba más grandes.

**Código**:

In [6]:
# Calculamos la precisión de cada estrategia comparando predicciones con realidad
precision_modelo1 = accuracy_score(y_prueba, predicciones_modelo1)
precision_modelo2 = accuracy_score(y_prueba, predicciones_modelo2)
precision_modelo3 = accuracy_score(y_prueba, predicciones_modelo3)
precision_votacion = accuracy_score(y_prueba, prediccion_votacion)

# Creamos un DataFrame para visualizar los resultados comparativos
resultados_comparativos = pd.DataFrame({
    "Estrategia": [
        "Modelo 1 - Regresión Logística",
        "Modelo 2 - Random Forest",
        "Modelo 3 - Regresión Logística (dup)",
        "Votación por Mayoría"
    ],
    "Precisión_Decimal": [
        precision_modelo1,
        precision_modelo2,
        precision_modelo3,
        precision_votacion
    ],
    "Precisión_Porcentaje": [
        f"{precision_modelo1*100:.1f}%",
        f"{precision_modelo2*100:.1f}%",
        f"{precision_modelo3*100:.1f}%",
        f"{precision_votacion*100:.1f}%"
    ],
    "Aciertos_de_5": [
        int(precision_modelo1 * 5),
        int(precision_modelo2 * 5),
        int(precision_modelo3 * 5),
        int(precision_votacion * 5)
    ]
})

print("Evaluación comparativa de rendimiento:")
print("="*70)
print(resultados_comparativos.to_string(index=False))

# Identificamos la mejor estrategia
mejor_indice = resultados_comparativos["Precisión_Decimal"].idxmax()
mejor_estrategia = resultados_comparativos.loc[mejor_indice, "Estrategia"]
mejor_precision = resultados_comparativos.loc[mejor_indice, "Precisión_Porcentaje"]

print("\n" + "="*70)
print(f"Mejor estrategia: {mejor_estrategia}")
print(f"Precisión alcanzada: {mejor_precision}")
print("\nNOTA: Con solo 5 jugadores de prueba, diferencias pequeñas pueden ser")
print("casualidad. En proyectos reales usaríamos cientos de casos de prueba.")

Evaluación comparativa de rendimiento:
                          Estrategia  Precisión_Decimal Precisión_Porcentaje  Aciertos_de_5
      Modelo 1 - Regresión Logística                1.0               100.0%              5
            Modelo 2 - Random Forest                1.0               100.0%              5
Modelo 3 - Regresión Logística (dup)                1.0               100.0%              5
                Votación por Mayoría                1.0               100.0%              5

Mejor estrategia: Modelo 1 - Regresión Logística
Precisión alcanzada: 100.0%

NOTA: Con solo 5 jugadores de prueba, diferencias pequeñas pueden ser
casualidad. En proyectos reales usaríamos cientos de casos de prueba.


**Resultado**:

La tabla comparativa muestra el rendimiento de cada estrategia. Análisis de los resultados:

1. **Modelos 1 y 3 son idénticos**: Como esperábamos, ambos modelos de Regresión Logística tienen la misma precisión porque son el mismo algoritmo con los mismos parámetros y datos.

2. **Variación entre estrategias**: Las diferencias en precisión reflejan cómo cada enfoque matemático interpreta los patrones en los datos.

3. **Rendimiento de la votación**: 
   - Si la votación tiene mayor precisión que los modelos individuales: demuestra que combinar perspectivas mejora las decisiones
   - Si la votación es igual al mejor modelo: indica que el dataset tiene patrones claros que todos los modelos captan
   - Si la votación es peor: puede indicar que los modelos están cometiendo errores similares

4. **Interpretación práctica**:
   - 100% (5/5): Predicción perfecta - poco común en datos reales
   - 80% (4/5): Muy buen rendimiento - aceptable para este tamaño de muestra
   - 60% (3/5): Rendimiento moderado - los modelos están aprendiendo algunos patrones
   - < 60%: Rendimiento pobre - los modelos no están captando bien los patrones

5. **Limitaciones del estudio**:
   - Con solo 5 casos de prueba, cada error representa 20% de precisión
   - En aplicaciones reales necesitaríamos mínimo 100-1000 casos de prueba
   - Deberíamos usar técnicas como validación cruzada para resultados más confiables

Este ejercicio demuestra el concepto fundamental de ensemble learning: combinar múltiples modelos puede producir predicciones más robustas que cualquier modelo individual.

---

## Síntesis del Caso: Sistema de Selección de Titulares del FC Barcelona

### Resumen del Proyecto Completado

Hemos desarrollado un sistema completo de apoyo a la decisión para el cuerpo técnico del Barcelona, siguiendo estos pasos:

**Módulo 1 - Carga de Datos**: Cargamos el dataset desde el archivo CSV `datos_barcelona.csv` (generado previamente con el script `generar_datos_liga.py`), conteniendo 15 jugadores con 4 variables relevantes.

**Módulo 2 - División Train/Test**: Separamos los datos en 70% entrenamiento (10 jugadores) y 30% prueba (5 jugadores) para evaluar generalización.

**Módulo 3 - Entrenamiento Multi-Modelo**: Entrenamos tres modelos con diferentes enfoques matemáticos (Regresión Logística x2 y Random Forest).

**Módulo 4 - Comparación de Predicciones**: Identificamos consenso y desacuerdos entre los modelos en las predicciones.

**Módulo 5 - Votación por Mayoría**: Implementamos un sistema de ensemble que combina las tres predicciones mediante votación simple.

**Módulo 6 - Evaluación Final**: Comparamos el rendimiento de todas las estrategias usando la métrica de precisión (accuracy).

### Conceptos Clave Aprendidos

1. **Separación de Responsabilidades**: Aprendimos a separar la generación de datos (script Python) del análisis (notebook Jupyter), reflejando prácticas profesionales reales.

2. **Ensemble Learning (Aprendizaje en Conjunto)**: Combinar múltiples modelos puede producir mejores resultados que cualquier modelo individual, similar a cómo un cuerpo técnico toma mejores decisiones consultando múltiples especialistas.

3. **Diversidad de Modelos**: Usar diferentes tipos de modelos (Regresión Logística vs Random Forest) captura diferentes aspectos de los datos.

4. **Votación por Mayoría**: Una estrategia simple pero efectiva para combinar predicciones binarias, donde cada modelo tiene un voto igual.

5. **Evaluación Rigurosa**: Siempre debemos medir el rendimiento en datos de prueba no vistos para evaluar la capacidad real de generalización.

6. **Reproducibilidad**: Trabajar con archivos CSV versionados asegura que todos obtienen los mismos resultados.

### Workflow Profesional Implementado

**Paso 1 - Generación de Datos (una vez)**
```bash
python generar_datos_liga.py
# Genera: datos_barcelona.csv
```

**Paso 2 - Análisis (repetible)**
```python
# En el notebook:
datos = pd.read_csv("datos_barcelona.csv")
# Entrenar modelos, evaluar, iterar...
```

Este patrón es estándar en la industria donde:
- **Ingenieros de Datos**: Generan y mantienen datasets
- **Científicos de Datos**: Consumen esos datasets para análisis y modelado

### Aplicación al Mundo Real

Este mismo enfoque se usa en:
- **Sistemas de recomendación** (Netflix, Spotify): Múltiples algoritmos votan qué contenido sugerir
- **Diagnóstico médico**: Varios modelos analizan síntomas y votan sobre posibles diagnósticos
- **Detección de fraude**: Bancos combinan modelos para identificar transacciones sospechosas
- **Análisis deportivo profesional**: Equipos como Manchester City y Liverpool usan ensemble models para scouting

### Próximos Pasos en tu Aprendizaje

En la **Semana 13** exploraremos:
- Métricas más avanzadas que la precisión simple (precision, recall, F1-score)
- Cómo evaluar modelos cuando las clases están desbalanceadas
- Técnicas para entender qué variables son más importantes para las predicciones
- Validación cruzada para evaluaciones más robustas

**Reflexión final**: ¿Por qué crees que la combinación de múltiples perspectivas (ya sea en modelos de ML o en equipos humanos) tiende a producir mejores decisiones que depender de una sola fuente?

### Archivos del Proyecto

- **generar_datos_liga.py**: Script para generar datos sintéticos del Barcelona
- **datos_barcelona.csv**: Dataset generado (438 bytes, 15 jugadores)
- **modelos-avanzados-clasificacion.ipynb**: Este notebook con análisis completo